# 05 · Fine-tune classifier trên ItW-train

Thích nghi miền có nhãn: fine-tune `FeatureTransformer` trên 165 video ItW-train. Kết quả mức đoạn: **44.1 ± 2.6** (3 seed), so với baseline 26.3.

Điểm chính:
- **Tăng cường ở mức feature** (không cần giải mã lại video): vị trí 18 feature ngẫu nhiên trong segment và crop không gian ngẫu nhiên (trái/giữa/phải).
- **Label smoothing 0.1** như lúc huấn luyện baseline gốc.
- **L2-SP**: phạt `||θ − θ_gốc||²` để không quên kiến thức cũ.
- Chọn `(lr, λ_L2SP)` trên **ItW-val**, rồi chạy cấu hình tốt nhất với 3 seed.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

# Notebook nằm trong thư mục notebooks/ -> thêm thư mục gốc repo vào sys.path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)

Repo root: <repo>


**`pipeline/s05b_finetune_itw.py` · phần import và hằng số**

`CROP_VARIANTS`: 3 bộ feature tương ứng crop không gian giữa / trái / phải.

In [2]:
from __future__ import annotations
import copy
import itertools
import json
import sys
from pathlib import Path
sys.path.insert(0, str(Path(ROOT / "pipeline/s05b_finetune_itw.py").resolve().parent.parent))
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tcfall.baseline import load_baseline
from tcfall.config import NUM_CLASSES, OUTPUTS, feature_dir
from tcfall.data import clean_labels, load_labels, load_split
from tcfall.metrics import core_metrics
from tcfall.segments import crop_positions, gather, window_start
OUT = OUTPUTS / "finetune"
CROP_VARIANTS = ("timefix", "timefix_start", "timefix_end")

**`pipeline/s05b_finetune_itw.py` · load_split_data**

Nạp mọi video của một split: feature của 3 crop không gian và mảng `start/end/label` của segment.

In [3]:
def load_split_data(split):
    labels, _ = clean_labels(load_labels("OOPS"))
    sp = load_split("OOPS")
    sub = labels[labels.path.map(sp) == split]
    vids = []
    for path, g in sub.groupby("path", sort=True):
        g = g.sort_values("start")
        feats = {v: np.load(feature_dir("OOPS", v) / f"{Path(path).name}.npy").astype(np.float32) for v in CROP_VARIANTS}
        vids.append({"path": path, "feats": feats, "start": g.start.to_numpy(), "end": g.end.to_numpy(),
                     "label": g.label.to_numpy()})
    return vids

**`pipeline/s05b_finetune_itw.py` · sample_batch**

Tạo một epoch dữ liệu. Khi `augment=True`, mỗi segment được lấy với **crop không gian ngẫu nhiên** và **vị trí thời gian ngẫu nhiên** `u ~ U(0,1)`; khi đánh giá thì cố định crop giữa, `u = 0.5`.

In [4]:
def sample_batch(vids, rng, augment: bool):
    X, Y = [], []
    for v in vids:
        for s, e, y in zip(v["start"], v["end"], v["label"]):
            var = CROP_VARIANTS[rng.integers(3)] if augment else "timefix"
            f = v["feats"][var]
            u = rng.random() if augment else 0.5
            X.append(gather(f, window_start(s, e, u, len(f)))); Y.append(y)
    return torch.from_numpy(np.stack(X)), torch.tensor(Y)

**`pipeline/s05b_finetune_itw.py` · predict**

Dự đoán theo segment, tuỳ chọn trung bình 3 crop không gian (`spatial`) và `k_temporal` lát thời gian (TTA); trả metric và dự đoán theo từng video.

In [5]:
@torch.no_grad()
def predict(model, vids, device, spatial=False, k_temporal=1):
    """Dự đoán theo segment; spatial=True: trung bình 3 crop không gian; k_temporal: TTA thời gian."""
    model.eval()
    ys, ps = [], []
    per_video = []
    for v in vids:
        P = 0
        variants = CROP_VARIANTS if spatial else ("timefix",)
        for var in variants:
            f = v["feats"][var]
            for u in crop_positions(k_temporal):
                X = torch.from_numpy(np.stack([gather(f, window_start(s, e, u, len(f)))
                                               for s, e in zip(v["start"], v["end"])])).to(device)
                P = P + torch.softmax(model(X)["logits"].float(), -1).cpu().numpy()
        pred = P.argmax(1)
        per_video.append((v["label"], pred))
        ys.append(v["label"]); ps.append(pred)
    return core_metrics(np.concatenate(ys), np.concatenate(ps)), per_video

**`pipeline/s05b_finetune_itw.py` · train**

Vòng huấn luyện:
1. nạp trọng số gốc, lưu bản sao `ref` để tính L2-SP;
2. mỗi epoch: tạo dữ liệu tăng cường, xáo trộn, chạy AdamW với cross-entropy (label smoothing 0.1) cộng `l2sp × Σ(θ − θ_gốc)²`, cắt gradient ở 1.0;
3. sau mỗi epoch đánh giá trên val với điểm `(Fallen-F1 + Fall∪Fallen-F1) / 2`, giữ checkpoint tốt nhất.

In [6]:
def train(base_state, train_vids, val_vids, lr, l2sp, seed, device, epochs=30, bs=32):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = load_baseline(device)
    model.load_state_dict(base_state)
    ref = {k: v.detach().clone() for k, v in model.named_parameters()}
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    best, best_state, best_ep = -1.0, None, 0
    for ep in range(epochs):
        model.train()
        X, Y = sample_batch(train_vids, rng, augment=True)
        perm = torch.randperm(len(Y), generator=torch.Generator().manual_seed(seed * 1000 + ep))
        for i in range(0, len(Y), bs):
            idx = perm[i:i + bs]
            logits = model(X[idx].to(device))["logits"]
            loss = F.cross_entropy(logits, Y[idx].to(device), label_smoothing=0.1)
            if l2sp > 0:
                loss = loss + l2sp * sum(((p - ref[n]) ** 2).sum() for n, p in model.named_parameters())
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        m, _ = predict(model, val_vids, device)
        sc = 0.5 * (m["fallen_f1"] + m["union_f1"])
        if sc > best:
            best, best_ep, best_state = sc, ep + 1, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, best, best_ep

**`pipeline/s05b_finetune_itw.py` · main**

1. Đo baseline gốc trên val.
2. Dò lưới 5 learning rate × 3 mức L2-SP (seed 42), lưu `finetune_grid_val.csv`.
3. Chạy cấu hình tốt nhất với seed 42, 43, 44; với mỗi seed đánh giá 4 kiểu (có/không TTA không gian, 1 hoặc 5 lát thời gian).
4. Lưu checkpoint (`baseline_ft_itw_seed*.pt`), bảng kết quả và trung bình ± độ lệch chuẩn.

In [7]:
def main():
    OUT.mkdir(parents=True, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tr, va, te = load_split_data("train"), load_split_data("val"), load_split_data("test")
    base = load_baseline(device)
    base_state = copy.deepcopy(base.state_dict())
    m0, _ = predict(base, va, device)
    print(f"Baseline gốc trên val: fallen={m0['fallen_f1']:.1f} union={m0['union_f1']:.1f}")

    rows = []
    for lr, l2sp in itertools.product((1e-5, 3e-5, 1e-4, 2e-4, 5e-4), (0.0, 1e-3, 1e-2)):
        _, sc, ep = train(base_state, tr, va, lr, l2sp, 42, device)
        rows.append({"lr": lr, "l2sp": l2sp, "val_score": round(sc, 2), "best_epoch": ep})
        print(rows[-1], flush=True)
    grid = pd.DataFrame(rows); grid.to_csv(OUT / "finetune_grid_val.csv", index=False)
    b = grid.loc[grid.val_score.idxmax()]
    lr, l2sp = float(b.lr), float(b.l2sp)
    print(f"Chọn lr={lr}, l2sp={l2sp}")

    res, preds = [], {}
    for seed in (42, 43, 44):
        model, sc, ep = train(base_state, tr, va, lr, l2sp, seed, device)
        ck = OUT / f"baseline_ft_itw_seed{seed}.pt"
        torch.save(model.state_dict(), ck)
        for spatial, k in ((False, 1), (True, 1), (False, 5), (True, 5)):
            mv, _ = predict(model, va, device, spatial, k)
            mt, pv = predict(model, te, device, spatial, k)
            tag = f"{'spatial' if spatial else 'center'}_tta{k}"
            preds[f"seed{seed}_{tag}"] = pv
            res.append({"seed": seed, "eval": tag, "best_epoch": ep, **{f"val_{x}": round(y, 2) for x, y in mv.items()},
                        **{f"test_{x}": round(y, 2) for x, y in mt.items()}})
            print(res[-1], flush=True)
    df = pd.DataFrame(res); df.to_csv(OUT / "finetune_results.csv", index=False)
    summ = df.groupby("eval")[[c for c in df if c.startswith(("val_", "test_"))]].agg(["mean", "std"]).round(2)
    summ.to_csv(OUT / "finetune_summary.csv")
    print(summ[[("val_fallen_f1", "mean"), ("test_fallen_f1", "mean"), ("test_fallen_f1", "std"),
                ("test_fall_f1", "mean"), ("test_union_f1", "mean"), ("test_union_f1", "std")]])
    import pickle
    with open(OUT / "finetune_test_preds.pkl", "wb") as f:
        pickle.dump(preds, f)
    (OUT / "finetune_config.json").write_text(json.dumps({"lr": lr, "l2sp": l2sp}, indent=2), encoding="utf-8")

## ▶ Chạy thật trên dữ liệu thật

Phần trên định nghĩa code. Từ đây là các ô chạy thật bằng dữ liệu, checkpoint và GPU của dự án.

Huấn luyện thật 2 epoch với cấu hình đã chọn (AdamW, L2-SP, tăng cường). Kết quả không ghi đè checkpoint.

In [8]:
import copy, json, torch
from tcfall.baseline import load_baseline

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = json.loads((OUT / "finetune_config.json").read_text(encoding="utf-8"))
print("Cấu hình đã chọn:", cfg)
tr, va = load_split_data("train"), load_split_data("val")
print("Số video:", len(tr), "train,", len(va), "val")
base = load_baseline(device)
m0, _ = predict(base, va, device)
print("Mô hình gốc trên val:", {k: round(v, 2) for k, v in m0.items()})
model, score, epoch = train(copy.deepcopy(base.state_dict()), tr, va, cfg["lr"], cfg["l2sp"], 42, device, epochs=2)
m1, _ = predict(model, va, device)
print(f"Sau 2 epoch: điểm chọn {score:.2f} ở epoch {epoch}:", {k: round(v, 2) for k, v in m1.items()})

Cấu hình đã chọn: {'lr': 0.0005, 'l2sp': 0.0}


Số video: 165 train, 81 val


Mô hình gốc trên val: {'fallen_f1': np.float64(28.57), 'fall_f1': np.float64(42.62), 'union_f1': np.float64(58.5), 'union_se': np.float64(52.86), 'union_sp': np.float64(85.5), 'bacc': np.float64(25.82), 'macro_f1': np.float64(26.71)}


Sau 2 epoch: điểm chọn 56.76 ở epoch 2: {'fallen_f1': np.float64(48.65), 'fall_f1': np.float64(50.81), 'union_f1': np.float64(64.86), 'union_se': np.float64(60.0), 'union_sp': np.float64(86.99), 'bacc': np.float64(24.0), 'macro_f1': np.float64(23.5)}


Nạp checkpoint fine-tune đầy đủ 30 epoch đã lưu, chấm lại trên val và so với bảng kết quả.

In [9]:
import pandas as pd

ft = load_baseline(device, str(OUT / "baseline_ft_itw_seed42.pt"))
m2, _ = predict(ft, va, device)
print("Checkpoint seed 42 trên val:", {k: round(v, 2) for k, v in m2.items()})
saved = pd.read_csv(OUT / "finetune_results.csv").query("seed == 42 and eval == 'center_tta1'").filter(like="val_")
print("Bảng đã lưu:\n", saved.to_string(index=False))

Checkpoint seed 42 trên val: {'fallen_f1': np.float64(52.5), 'fall_f1': np.float64(58.59), 'union_f1': np.float64(71.94), 'union_se': np.float64(71.43), 'union_sp': np.float64(85.87), 'bacc': np.float64(26.72), 'macro_f1': np.float64(27.26)}
Bảng đã lưu:
  val_fallen_f1  val_fall_f1  val_union_f1  val_union_se  val_union_sp  val_bacc  val_macro_f1
          52.5        58.59         71.94         71.43         85.87     26.72         27.26
